In [ ]:
# Copyright 2026 International Business Machines

# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at

#  http://www.apache.org/licenses/LICENSE-2.0

# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Terrakit: Labels to dataset pipeline for Salt Marsh dataset preparation

This notebook demonstrates generating a ML-ready dataset from a collection of salt marsh labels using [TerraKit](https://torchgeo.org/terrakit/), an open source python library purpose built for generating ML-ready geospatial datasets. 

The labelled data used in this notebook have been derived from the Environment Agency Saltmarsh Extent and Zonation dataset. Here, we work with a small area of saltmarsh in the Dee Estuary, using 10m resolution labels derived from the original vector data.

TerraKit extracts date and location information from each label file. This information is then used by TerraKit to download the corresponding satellite image from the data source requested, which in this case is [Sentinel-2 Level-2A](https://docs.sentinel-hub.com/api/latest/data/sentinel-2-l2a/). The generated dataset will consist of data/label pairs that have been chipped (split) into 256x256 images and is ready to use by TerraTorch to train a geospatial foundation model. 


## 0. Setup

### 0.1 Check Python Version

It's recommended that you run this notebook using Python 3.12.

In [ ]:
!python --version

### 0.2 Install and Import Dependencies

If running locally, ensure you have the necessary packages installed. This assumes you're in the project root directory. 

Once installed, import dependencies.

In [ ]:
import sys
%pip install uv
if 'google.colab' in sys.modules:
  !uv pip install --system git+https://github.com/torchgeo/terrakit.git
else:
  !uv pip install git+https://github.com/torchgeo/terrakit.git gdown

In [ ]:
# Import dependencies
import gdown
import os

# Import TerraKit
import terrakit

Check you are using the latest version of TerraKit. This notebook has been tested with 0.1.8.post1.

In [ ]:
!terrakit --version

### 0.3 Download Salt Marsh Label

The following citation applies to the downloaded file:

> Citation
> The extent and zonation of saltmarsh in England: 2016-2019. Environment Agency, (2022) https://www.gov.uk/government/publications/the-extent-and-zonation-of-saltmarsh-in-england-2016-2019 
>
> Attribution
> © Environment Agency copyright and/or database right 2023.
> Licensed under the Open Government Licence v3.0.

In [ ]:

os.makedirs("./data/labels", exist_ok=True)

file_id = "1bq3GpU242zXZcim4BBSVM9nWD2kLoZny"
output = "./data/labels/RGBN_dee_2024_res_10m_lab.tif"

if not os.path.isfile(output):
    gdown.download(
        f"https://drive.google.com/uc?id={file_id}",
        output,
        quiet=False
    )

## 1. Process labels
The first step in the TerraKit dataset curation pipeline is processing the labels. 

Use `process_labels()` to generate two DataFrames, where the first contains bound box and temporal information for all of the geospatial locations identified in the labels files, and the second containing the label geometries. Each DataFrame is also saved as a .shp file in the working directory.

The `process_labels` function takes the following arguments:
- `labels_folder`: The directory containing the labels.
- `dataset_name`: The name of the dataset.
- `working_dir`: The working directory where the processed labels will be saved.
- `label_type`: The type of labels, either "raster" or "vector".
- `datetime_info`: The format of the datetime information in the labels.


The `labels_folder` should also include a `metadata.csv` which will contain temporal information about each label file. As the salt marsh label files have a unique format, we will generate this `metadata.csv` file by extracting the temporal information from each label filename.



In [ ]:
# Set a working directory, a dataset name and a directory where some labels can be found
DATASET_NAME = "s2_saltmarsh_extent" # dataset_name
WORKING_DIR = f"./tmp/{DATASET_NAME}" # This is where your new dataset will be saved to.
LABELS_FOLDER = "./data/labels" # labels_folder

In [ ]:
# Add the filename and date information to `metadata.csv` in the `labels_folder`. This will help Terrakit know which label files to processs.
!echo "filename,date" > $LABELS_FOLDER/metadata.csv
!echo "RGBN_dee_2024_res_10m_lab.tif,2024-06-15" >> $LABELS_FOLDER/metadata.csv

In [ ]:
# Process the labels by providing a labels folder, working directory and dataset name. 
# The resulting shapefiles will be saved in the working directory.
labels_gdf, grouped_bbox_gdf = terrakit.process_labels(
    labels_folder=LABELS_FOLDER,
    dataset_name=DATASET_NAME,
    working_dir=WORKING_DIR,
    label_type="raster", # Can be raster or vector labels
    datetime_info="csv" # Datetime information is provided in csv format
)

In [ ]:
# Plot the process labels and bboxes to confirm they appear as expected.
# This helps to validate that the label geometries and bounding boxes have been extracted correctly.
# Here you can also see the bounding boxes which will be used by `download_data()` to download data for in the next step.
terrakit.plotting.plot_label_dataframes(labels_gdf, grouped_bbox_gdf)

## 2. Download the data

Once labels have been processed, next up in the TerraKit pipeline is downloading the data.

Use `download_data()` to download data from a set of data connectors for a time and location specified by the shapefiles saved by the process_labels pipeline step. Terrakit assumes that the shapefiles are in the working directory (which is where the process_labels pipeline step saves them to).

TerraKit allows you to request data is downloaded from any number of available data connectors, enabling you to download data from multiple sources in a single step. Here we download data solely from the S2L2A collection using the Sentinel-AWS data connector. This is the only data connector that does not require authentication, making it super simple to use.

In [ ]:
config = {
    "download": {
        "data_sources": [
            {
                "data_connector": "sentinel_aws",
                "collection_name": "sentinel-2-l2a",
                "bands": ["blue", "green", "red", "nir08", "swir16", "swir22"],
            },
        ],
        "date_allowance": {"pre_days": 30, "post_days": 30}, # Define how long before and after the date specified in the shapefiles.
        "transform": {
            "scale_data_xarray": False,
            "impute_nans": False,
            "reproject": False,
        },
        "max_cloud_cover": 80, # Set the maximum percent cloud cover allowed for the data to be downloaded.
        },
    }

queried_data = terrakit.download_data(
    dataset_name=DATASET_NAME,
    working_dir=WORKING_DIR,
    data_sources=config["download"]["data_sources"],
    date_allowance=config["download"]["date_allowance"],
    transform=config["download"]["transform"],
    max_cloud_cover=config["download"]["max_cloud_cover"],
    keep_files=True, # Set to True to keep any downloaded files that were created ahead of any transformations.
)

### 2.2: Inspect the data
Use `terrakit.plot_tiles_and_label_pair()` to inspect the downloaded tiles and corresponding labels.

In [ ]:
terrakit.plotting.plot_tiles_and_label_pair(
    queried_data, bands=["red", "green", "blue"], scale = 2
)

## 3. Chip the data
Now that the tiled data has been downloaded, let's chip it accordingly.

### 3.1: Chip the data and label pairs
Use `chip_and_label_data()` to chip (split) the data and label pairs into 256x256 samples. The output will be saved in the `WORKING_DIR` with the specified `chip_suffix` and `chip_label_suffix`. 

In [ ]:
res = terrakit.chip_and_label_data(
    dataset_name=DATASET_NAME,
    working_dir=WORKING_DIR,
    queried_data=queried_data, # data to chip (returned from download_data())
    sample_dim=256, # chip dimensions (256x256)
    data_suffix=".tif", # data file suffix (default: ".tif")
    label_suffix="_label.tif", # label file suffix (default: "_label.tif")
    chip_suffix="_chip.tif", # output S2 chip suffix
    chip_label_suffix="_chip_lab.tif" # Output label chip suffix
)

### 3.2 Check the results
Use `plot_chip_and_label_pairs` to check the chip and label pairs look as expected.

In [ ]:
terrakit.plotting.plot_chip_and_label_pairs(
    res, bands=["red", "green", "blue"], samples=10,
    chip_suffix="_chip.tif",
    chip_label_suffix="_chip_lab.tif",
)

## 4. Next steps
Congratulations! You have successfully created an ML-ready dataset for training a salt marsh extent model using S2 data for the Sentinel AWS achieve. Next steps is to see if you can prepare your own dataset. Using TerraKit's hugging_face_file_downloader function to grab some example labels from the `ibm-nasa-geospatial/hls_burn_scars` huggingface repo of recent wildfire events. You could also try out using your own labels.

### 0. Download example labels from Hugging Face

<div class="alert alert-block alert-success">
<b>Example labels:</b> To download a set of example raster labels, use the `hugging_face_file_downloader` function.
</div>

In [ ]:
from glob import glob
from pathlib import Path

from terrakit.general_utils.labels_downloader import hugging_face_file_downloader

DATASET_NAME_WILDFIRE = "burn_scars"
WORKING_DIR_WILDFIRE = f"./tmp/{DATASET_NAME_WILDFIRE}"
LABELS_FOLDER_WILDFIRE = "./data/burn_scar_labels"

example_wildfire_labels = [
    "subsetted_512x512_HLS.S30.T10SEH.2018245.v1.4.mask.tif",
    "subsetted_512x512_HLS.S30.T10SGD.2021306.v1.4.mask.tif",
]

for filename in example_wildfire_labels:
    hugging_face_file_downloader(
        repo_id="ibm-nasa-geospatial/hls_burn_scars",
        filename=filename,
        revision="e48662b31288f1d5f1fd5cf5ebb0e454092a19ce",
        subfolder="training",
        dest=LABELS_FOLDER_WILDFIRE,
    )

### 1. Process the labels to extract location/datetime info and return a geodataframe
As before, this initial steps takes a directory containing some label files. This time the  date is assumed to be contained in the filename. Supported date types are `YYYYDDD (7), YYYYMMDD (8), YYMMDD (6 -> 20YYMMDD).` Checkout the TerraKit API docs for help: https://torchgeo.org/terrakit/api/labels/#terrakit.transform.labels.process_labels. There is also some helpful info on how to `process_labels()` works here: https://torchgeo.org/terrakit/process_labels/#datetime-information

In [ ]:
# Your code here
# ------------------------------------------------------------
# labels_gdf, grouped_bbox_gdf = terrakit.process_labels(
#     labels_folder=# fill in here,
#     dataset_name=# fill in here,
#     working_dir=# fill in here,
#     label_type="raster", # update to "vector" if using vector data for labels
#     datetime_info="filename", # Tells Terrakit to look in the filename to find a datetime string for each label file. Update to "csv" if needed.
# )
# ------------------------------------------------------------

<details>
<summary><b>Reveal solution</b></summary>
```
labels_gdf, grouped_bbox_gdf = terrakit.process_labels(
    labels_folder=LABELS_FOLDER_WILDFIRE,
    dataset_name=DATASET_NAME_WILDFIRE,
    working_dir=WORKING_DIR_WILDFIRE,
    label_type="raster",
)
```
</details>

## 2. Download and plot the data

Learn more about `download_data()` here: https://torchgeo.org/terrakit/download_data/

Check out the API docs for `download_data()` here: https://torchgeo.org/terrakit/api/download/

In [ ]:
## Your code here
## ------------------------------------------------------------
# config = {
# }

# queried_data = terrakit.download_data(
#     data_sources=
#     date_allowance=
#     transform=
#     max_cloud_cover=
#     dataset_name=DATASET_NAME_WILDFIRE,
#     working_dir=WORKING_DIR_WILDFIRE,
#     keep_files=False, # Remove the files generated in the previous step. It can be useful to remove these as they are no longer needed.
# )
## ------------------------------------------------------------

In [ ]:
## Your code here
## ------------------------------------------------------------
# terrakit.plotting.plot_tiles_and_label_pair(
#     queried_data, bands=
# )
## ------------------------------------------------------------

<details>
<summary><b>Reveal solution</b></summary>


    config = {

        "download": {
            "data_sources": [
                {
                    "data_connector": "sentinel_aws",
                    "collection_name": "sentinel-2-l2a",
                    "bands": ["blue", "green", "red"],
                },
            ],
            "date_allowance": {"pre_days": 0, "post_days": 21},
            "transform": {
                "scale_data_xarray": True,
                "impute_nans": True,
                "reproject": True,
            },
            "max_cloud_cover": 80,
        },
    }

    queried_data = 
        terrakit.download_data(
    
            data_sources=config["download"]["data_sources"],
            date_allowance=config["download"]["date_allowance"],
            transform=config["download"]["transform"],
            max_cloud_cover=config["download"]["max_cloud_cover"],
            dataset_name=DATASET_NAME_WILDFIRE,
            working_dir=WORKING_DIR_WILDFIRE,
            keep_files=False,
    )

    terrakit.plotting.plot_tiles_and_label_pair(
        queried_data, bands=config["download"]["data_sources"][0]["bands"]
    )
</details>

## 3. Chip the data
Now that the tiled data has been downloaded, let's chip it accordingly.

Check out the API docs for `chip_and_label_data()` here: https://torchgeo.org/terrakit/api/chip/

In [ ]:
# # Your code here
## ------------------------------------------------------------
# chip_sample_dimensions = 

# res = terrakit.chip_and_label_data(
#     dataset_name=DATASET_NAME_WILDFIRE,
#     sample_dim=chip_sample_dimensions,
#     queried_data=queried_data,
#     working_dir=WORKING_DIR_WILDFIRE,
# )

# terrakit.plotting.plot_chip_and_label_pairs(
#     res, 
#     bands= 
#     samples=
# )
## ------------------------------------------------------------

<details>
<summary><b>Reveal solution</b></summary>
```
    
    chip_args = {
        "chip": {"sample_dim": 256},
    }

    res = terrakit.chip_and_label_data(
        dataset_name=DATASET_NAME_WILDFIRE,
        sample_dim=chip_args["chip"]["sample_dim"],
        queried_data=queried_data,
        working_dir=WORKING_DIR_WILDFIRE,
    )

    terrakit.plotting.plot_chip_and_label_pairs(
       res, bands=config["download"]["data_sources"][0]["bands"], samples=10
    )
</details>

<div class="alert alert-block alert-info">
Now see if you can try with your own labels. For help, take a look at the <a href="https://torchgeo.org/terrakit/examples/labels_to_data/#raster-labels-to-data">TerraKit documentation</a>.
</div>

Next steps is to see how well this dataset does at training a model following the steps in the Fine Tuning with Terramind notebook. See if you can improve the model performance by using multi-modal data by adding another data connector to `download_data()`.